# Tutorial 9: Fine-Tuning a Pretrained Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/tutorials/09-fine-tuning-nllb.ipynb)

**Reading. About 30 minutes, plus the training you choose; the training cell prints its own estimate.**

Every model you have built so far started from nothing and learned your language from your
A8 pairs alone. Most working systems, and most Final Projects, start somewhere else: from a
model already trained on many languages, which is then trained a little more on your pairs.
That second step is **fine-tuning**. This notebook fine-tunes Meta's NLLB-200, the model A12
and A13 already called, on your A8 split, and measures what it buys: against the same model
off the shelf, and against your A9 model, which saw the same pairs but started from scratch.

By the end you will be able to:

1. score a pretrained model off the shelf on your own test set, as the number fine-tuning has
   to beat;
2. say how NLLB's language codes tell one model which language to read and which to write,
   having built the code-tagged sequences yourself, and choose a covered language to borrow
   when yours is not one of NLLB's, with a reason you can test;
3. fine-tune the model on your pairs within a step budget you choose, resumably, keeping the
   checkpoint with the best validation loss;
4. say what fine-tuning bought over the model off the shelf and over a model trained from
   scratch, and where in the output the difference shows.

These serve Lecture 14's outcome *explain tagging and why zero-shot works at all*: NLLB's
language codes are the tags of A14, at the scale of 200 languages. They are also the first
step of most Final Projects.

**Running in Google Colab?** Runtime → Change runtime type → an **A100, L4 or G4** GPU, not
the T4, which has too little memory; then Run all. It stops at each TODO and at the
prediction until you write it. It runs on a built-in English–Spanish sample until you change
two lines in Step 1. When you finish, disconnect the GPU: Runtime → Disconnect and delete
runtime. The last cell says why.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**TorchLingo tutorial** — assigned reading for Lecture 14 (*Multilingual NMT and "Zero-shot" NMT*).

In [ ]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
import torchlingo


In [ ]:
# Check the setup and connect Google Drive, where this notebook keeps its checkpoints and output.
from torchlingo.colab import course_dir, setup

env = setup(gpu=True, drive=True)       # stops with instructions if there is no GPU
device = env.device
OUT_DIR = course_dir(env.drive_dir, "tutorial9", create=True)   # MyDrive/CS479/tutorial9

import json
import random
from pathlib import Path

import pandas as pd
import torch
import torchlingo


## Step 1: your A8 pairs

Fine-tuning trains on your A8 training set, keeps the checkpoint that does best on its
validation set, and is scored on its test set: the same split A8, A9 and A13 used, so the
scores line up with theirs. Set `LANGUAGE` to your A8 language, as Learning Suite's data
folder names it ("Malagasy", "Chinese T"), and `A8_SPLIT` to the folder the A8 kickoff
notebook wrote your split to, together. With neither set, the notebook runs on a built-in
English–Spanish sample.

If your A9 translations are there too (`test-a9.hyp`, which Lecture 9's Part B wrote), the
report at the end sets your A9 model beside NLLB.

In [ ]:
LANGUAGE = "Spanish"    # <- your A8 language, e.g. "German", "Malagasy", "Chinese T"
A8_SPLIT = None         # <- e.g. env.drive_dir / "CS479" / "assignment8"

SEED = 479


In [ ]:
# Setup: run this cell, no need to read it. It loads your split, or the sample.
if (A8_SPLIT is None) != (LANGUAGE == "Spanish"):
    raise SystemExit(
        "Set LANGUAGE and A8_SPLIT together: both for your A8 language, or neither for the "
        "built-in Spanish sample."
    )


def read_split(name):
    return pd.read_csv(Path(A8_SPLIT) / name, sep="\t", dtype=str, keep_default_na=False)


A9_HYP = None
if A8_SPLIT is None:
    from torchlingo.colab import fetch_data

    (sample,) = fetch_data(["data/example.tsv"])
    pairs = pd.read_csv(sample, sep="\t", dtype=str, keep_default_na=False)
    words = pairs.src.str.split().str.len()
    pairs = pairs[(pairs.kind == "transcript") & words.between(4, 40)][["src", "tgt"]]
    pairs = pairs.drop_duplicates("src").sample(frac=1, random_state=SEED)
    train, val, test = pairs[:4000], pairs[4000:4200], pairs[4200:4400]
    print("*** The built-in English-Spanish sample. Set LANGUAGE and A8_SPLIT in the cell above")
    print("*** to your A8 language and split, then run all again. ***\n")
else:
    A8_SPLIT = course_dir(
        env.drive_dir, Path(A8_SPLIT).name, needs=["train.tsv", "val.tsv", "test.tsv"],
        hint="The A8 kickoff notebook writes them; run it that far first.",
    )
    train, val, test = (read_split(f"{n}.tsv") for n in ("train", "val", "test"))
    if (Path(A8_SPLIT) / "test-a9.hyp").is_file():
        A9_HYP = Path(A8_SPLIT) / "test-a9.hyp"

# A language written without spaces is scored as A9 scored it, with spBLEU's tokenizer.
NO_SPACES = test.tgt.str.split().str.len().mean() < 3
TOKENIZE = "flores200" if NO_SPACES else "13a"
print(f"{LANGUAGE}: {len(train):,} training, {len(val):,} validation and {len(test):,} test pairs.")
print("Your A9 translations: " + (str(A9_HYP) if A9_HYP else "not found, so no A9 row"))


## Step 2: one model, 200 languages, and the codes that steer it

NLLB-200 is one encoder–decoder Transformer, the architecture you built in A8, trained on
about 200 languages at once. Nothing in its weights says "English to Spanish". What steers it
is a **language code** at the front of each sequence, the same idea as A14's `<2x>` tags:

- the **source** is the source language's code, then the sentence's pieces, then `</s>`;
- the **target** is the target language's code, then the translation's pieces, then `</s>`.

So `eng_Latn Good morning </s>` paired with `spa_Latn Buenos días </s>` teaches English to
Spanish, and the same model, given `spa_Latn` at the front of the source and `eng_Latn` at the
front of the target, translates the other way. When it translates, the decoder is handed the
target code as its first word (`forced_bos_token_id` in the code A12 and A13 used), and
continues from there. That is all "choosing the language" means.

The codes are written as three letters for the language and four for its script: `eng_Latn`,
`tel_Telu`, `zho_Hant` for Traditional Chinese.

**Yours to write: `with_codes`**, two cells down. Given the piece ids of a source sentence and
of its translation, the two codes' ids and the id of `</s>`, return the two sequences the
model trains on, as lists. The cell checks it on a case you can work out by hand, then
against what NLLB's own tokenizer builds for a real pair: if they agree, you have written
down exactly what the model sees.

The next cell loads NLLB-200, the 600M-parameter distilled model: about 2.4 GB, a minute or
two to download.

In [ ]:
# Setup: run this cell, no need to read it. It loads NLLB-200 and its tokenizer.
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

from torchlingo_harness.nllb import NLLB_MODEL, NLLBTranslator

tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL, src_lang="eng_Latn")
# Trained in float32; the training cell runs the arithmetic in bfloat16 where the GPU has it.
model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL, dtype=torch.float32).to(device)
NLLB_CODES = {t for t in tokenizer.additional_special_tokens if len(t) == 8 and t[3] == "_"}
print(f"{NLLB_MODEL}: {sum(p.numel() for p in model.parameters()):,} parameters, "
      f"{len(NLLB_CODES)} language codes")


In [ ]:
def with_codes(source_ids, target_ids, source_code_id, target_code_id, end_id):
    """Return (input_ids, labels): the source and target sequences NLLB trains on, as lists.

    TODO: write it, as Step 2 describes.
    """
    raise NotImplementedError("write with_codes, then run this cell again")


# A case you can work out by hand: made-up ids for two source pieces and one target piece.
got = with_codes([7, 8], [9], source_code_id=100, target_code_id=200, end_id=2)
expected = ([100, 7, 8, 2], [200, 9, 2])
assert tuple(map(list, got)) == expected, f"for this case it should be {expected}; yours returned {got}"

# The same, against NLLB's own tokenizer on a real pair.
english, spanish = "Good morning, my friends.", "Buenos días, amigos míos."
tokenizer.src_lang, tokenizer.tgt_lang = "eng_Latn", "spa_Latn"
theirs = tokenizer(english, text_target=spanish)
pieces = lambda text: tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text))
mine = with_codes(pieces(english), pieces(spanish), tokenizer.convert_tokens_to_ids("eng_Latn"),
                  tokenizer.convert_tokens_to_ids("spa_Latn"), tokenizer.eos_token_id)
assert list(mine[0]) == theirs["input_ids"] and list(mine[1]) == theirs["labels"], (
    f"yours differ from NLLB's tokenizer:\n  yours  {list(mine[0])}  {list(mine[1])}\n"
    f"  NLLB's {theirs['input_ids']}  {theirs['labels']}")
print("Yours match NLLB's tokenizer. What the model sees, as pieces:")
print("  source:", tokenizer.convert_ids_to_tokens(theirs["input_ids"]))
print("  target:", tokenizer.convert_ids_to_tokens(theirs["labels"]))


## Step 3: is your language one of NLLB's?

NLLB covers about 200 languages, most of the course's among them. Of the A8 languages it does
**not** cover Hmong; of the A12 languages, Efik, Kosraean, Palauan, Pohnpeian and Yapese. The
next cell says which yours is.

**If NLLB covers your language,** its code is the target code, and there is nothing to decide.

**If it does not,** you still need a target code, since the model must be told what to write,
and you choose one of NLLB's languages to **borrow**. Fine-tuning then teaches the model that
this code now means your language. It works because the code is only a learned starting point:
a related language's code puts the decoder near the right script, spelling and word order
before your pairs say anything, so fine-tuning has less to move. What it costs: the model
keeps some of the borrowed language's habits (its words, its spelling conventions) until your
pairs outweigh them, and the borrowed language's own translations get worse in the bargain.

Choose by reasoning first, then test the reasoning. What to weigh:

- **language family**: a relative shares grammar and often words;
- **script**: the same alphabet means the same pieces, and a different script means none;
- **shared vocabulary or loanwords**, and the languages yours borrowed them from;
- **word order**: subject, verb and object in the same order help most;
- **region and contact**: neighbors' languages, and the language of the region's schools and
  church.

Then list two or three candidate codes in `CANDIDATES` (find them in the
[FLORES-200 list](https://github.com/facebookresearch/flores/blob/main/flores200/README.md)).
The cell translates a dozen of your test sentences into each candidate, off the shelf, and
prints the chrF of each against your references. chrF counts matching characters, so even a
related language's output scores above an unrelated one's. A higher chrF is evidence for a
candidate, not the whole answer: say in `REASON` what you weighed, then set `BORROW`.

In [ ]:
from torchlingo.evaluation import compute_bleu, compute_chrf

# The course's languages and their NLLB codes; None where NLLB does not cover the language.
COURSE_CODES = {
    "Chinese T": "zho_Hant", "French": "fra_Latn", "Georgian": "kat_Geor", "German": "deu_Latn",
    "Greek": "ell_Grek", "Hmong": None, "Japanese": "jpn_Jpan", "Khmer": "khm_Khmr",
    "Korean": "kor_Hang", "Lao": "lao_Laoo", "Malagasy": "plt_Latn", "Malay": "zsm_Latn",
    "Norwegian": "nob_Latn", "Portuguese": "por_Latn", "Romanian": "ron_Latn",
    "Russian": "rus_Cyrl", "Spanish": "spa_Latn", "Tagalog": "tgl_Latn", "Thai": "tha_Thai",
    "Telugu": "tel_Telu", "Efik": None, "Kosraean": None, "Palauan": None, "Pohnpeian": None,
    "Yapese": None,
}
assert LANGUAGE in COURSE_CODES, f"LANGUAGE must be one of {sorted(COURSE_CODES)}"
COVERED = COURSE_CODES[LANGUAGE]
assert COVERED is None or COVERED in NLLB_CODES, f"{COVERED} is not in this NLLB's list"

CANDIDATES = []   # <- if NLLB does not cover your language: two or three codes, e.g. ["abc_Latn", ...]


def chrf_into(code, sentences, references):
    """chrF of NLLB off the shelf, translating sentences into code, against references."""
    tokenizer.src_lang = "eng_Latn"
    hyps = NLLBTranslator(model.eval(), tokenizer, code, device, NLLB_MODEL)(sentences, batch_size=12)
    return compute_chrf(hyps, references).score, hyps


if COVERED:
    print(f"NLLB covers {LANGUAGE}: its code is {COVERED}. Nothing to borrow; go on to Step 4.")
else:
    assert CANDIDATES, (f"NLLB does not cover {LANGUAGE}. List two or three candidate codes in "
                        "CANDIDATES, as Step 3 describes, then run this cell again.")
    unknown = [c for c in CANDIDATES if c not in NLLB_CODES]
    assert not unknown, f"not NLLB codes: {unknown}; codes look like 'tgl_Latn'"
    dozen = test.sample(12, random_state=SEED)
    print(f"NLLB does not cover {LANGUAGE}. chrF of each candidate, off the shelf, on 12 of your")
    print("test sentences (higher means closer to your references, character by character):\n")
    for code in CANDIDATES:
        score, hyps = chrf_into(code, list(dozen.src), list(dozen.tgt))
        print(f"  {code}   chrF {score:5.1f}   e.g. {hyps[0][:70]!r}")
    print(f"\n  reference:          {dozen.tgt.iloc[0][:70]!r}")

In [ ]:
BORROW = ""   # <- if NLLB does not cover your language: the code you borrow, from CANDIDATES
REASON = ""   # <- why: family, script, vocabulary, word order, region; and what the chrF said

if COVERED:
    TARGET_CODE, REASON = COVERED, f"{LANGUAGE} is one of NLLB's languages"
else:
    assert BORROW in CANDIDATES, "Set BORROW to one of your CANDIDATES, then run this cell again."
    assert REASON.strip(), "Say in REASON why you chose it, then run this cell again."
    TARGET_CODE = BORROW
tokenizer.src_lang, tokenizer.tgt_lang = "eng_Latn", TARGET_CODE
print(f"Target code: {TARGET_CODE}  ({REASON.strip()})")


## Step 4: the number to beat

Before training anything, score NLLB as it comes, on your whole test set, decoded greedily
and with a beam of 5, as A13 decoded. For a language NLLB covers, this is a real system's
score; for a borrowed code, it is the score of translating into the borrowed language, and
should be low. Either way it is what fine-tuning has to beat. The translations are saved in
`MyDrive/CS479/tutorial9`, and a rerun reuses them.

In [ ]:
# Setup: run this cell, no need to read it. It decodes and scores the test set.
sources, references = list(test.src), list(test.tgt)
DECODINGS = {"greedy": dict(decode_strategy="greedy", batch_size=32),
             "beam 5": dict(decode_strategy="beam", beam_size=5, batch_size=16)}
scores = []


def decode(system, how, reuse):
    """Translate the test set with the model as it is now; save it as <system>-<how>.hyp."""
    path = OUT_DIR / f"{system}-{how.replace(' ', '')}.hyp"
    if reuse and path.is_file():
        hyps = path.read_text(encoding="utf-8").split("\n")[: len(sources)]
        if len(hyps) == len(sources):
            return hyps
    tokenizer.src_lang = "eng_Latn"
    hyps = NLLBTranslator(model.eval(), tokenizer, TARGET_CODE, device, NLLB_MODEL)(
        sources, **DECODINGS[how])
    path.write_text("\n".join(hyps) + "\n", encoding="utf-8")
    return hyps


def score(system, how, hyps):
    bleu, chrf = compute_bleu(hyps, references, tokenize=TOKENIZE), compute_chrf(hyps, references)
    scores.append({"system": system, "decoding": how, "bleu": round(bleu.score, 2),
                   "chrf": round(chrf.score, 2), "bleu_signature": str(bleu.signature)})
    print(f"{system:<13} {how:<7} BLEU {bleu.score:6.2f}   chrF {chrf.score:6.2f}")


off_the_shelf = {how: decode("off-the-shelf", how, reuse=True) for how in DECODINGS}
for how, hyps in off_the_shelf.items():
    score("off the shelf", how, hyps)
if A9_HYP:
    score("A9, scratch", "greedy", A9_HYP.read_text(encoding="utf-8").split("\n")[: len(sources)])


## Step 5: a budget, and a prediction

Fine-tuning is training, measured here in **steps**: each step updates the weights once, from
`BATCH_SIZE` × `ACCUMULATE` pairs. Gradient accumulation adds up the gradients of several
small batches before one update, so the update sees as many pairs as one large batch would,
in the memory of a small one. The default, 2,000 steps of 32 pairs, is 64,000 pairs, about
two-thirds of a 100,000-pair A8 training set, once. More steps cost time and, past some
point, buy nothing: the validation loss, checked every `EVAL_EVERY` steps, says when.

The learning rate is much smaller than A8's. The weights are already good; fine-tuning should
nudge them toward your language, not overwrite what 200 languages taught them.

**Write a prediction first**, in `PREDICTION`: by how much will fine-tuning raise BLEU and
chrF over the model off the shelf, and will the fine-tuned model beat your A9 model? Say why,
in a sentence or two.

In [ ]:
STEPS = 2000          # <- your budget; the training cell prints how long it will take
BATCH_SIZE = 16       # pairs per forward pass; lower it if the GPU runs out of memory
ACCUMULATE = 2        # batches per update: 16 x 2 = 32 pairs per step
LEARNING_RATE = 1e-4  # Adafactor's usual rate for fine-tuning; see Step 6
WARMUP = 100          # steps over which the learning rate rises from zero
EVAL_EVERY = 250      # steps between validation checks

PREDICTION = ""   # <- your prediction, in a sentence or two

assert PREDICTION.strip(), "Write your prediction in PREDICTION, then run this cell again."
print(f"{STEPS:,} steps of {BATCH_SIZE * ACCUMULATE} pairs: {STEPS * BATCH_SIZE * ACCUMULATE:,} "
      f"pairs, {STEPS * BATCH_SIZE * ACCUMULATE / len(train):.2f} passes over your training set.")


## Step 6: fine-tune

The training loop is A8's, with four differences worth reading for:

- **The loss comes from the model.** Given `labels`, a Hugging Face model computes the
  cross-entropy itself; padding in the labels is marked `-100`, which it skips.
- **Accumulation.** The loss of each small batch is divided by `ACCUMULATE` and its gradients
  added up, and only then does the optimizer step.
- **Mixed precision.** On an A100, L4 or G4, the arithmetic runs in bfloat16, about twice as
  fast, while the weights stay in float32.
- **The optimizer is Adafactor, not A8's Adam.** Adam keeps two running averages for every
  weight, so for 615 million weights a checkpoint would be about 7 GB, and a run keeps two of
  them on your Drive. Adafactor keeps a few numbers per row and column of each weight matrix
  instead, so a checkpoint is about 2.5 GB, mostly the weights themselves. It is what NLLB's
  own fine-tuning recipes use.

As in A8 and A9, the run checkpoints to Drive every ten minutes and keeps the checkpoint with
the best validation loss. That takes about **8 GB free on your Drive**; the cell checks first,
and stops with a message if there is not. **If the session drops, run the setup cells and this
one again**: it resumes where it stopped. Run it after it has finished and it skips straight to
the best checkpoint. Don't disconnect while it runs.

In [ ]:
import contextlib
import shutil
import time
from functools import lru_cache

from transformers.optimization import Adafactor

from torchlingo.training_checkpoint import TrainingCheckpointer


def batch_of(frame):
    """Tokenize a frame of pairs with the language codes; padding in the labels becomes -100."""
    batch = tokenizer(list(frame.src), text_target=list(frame.tgt), max_length=128,
                      truncation=True, padding=True, return_tensors="pt")
    batch["labels"][batch["labels"] == tokenizer.pad_token_id] = -100
    return batch.to(device)


@lru_cache(maxsize=1)
def pass_order(n):
    """The training pairs in the order of pass n over them, seeded, so a resumed run repeats it."""
    return train.sample(frac=1, random_state=SEED + n)


def micro_batch(k):
    """The k-th batch of training pairs."""
    per_pass = len(train) // BATCH_SIZE
    start = (k % per_pass) * BATCH_SIZE
    return batch_of(pass_order(k // per_pass)[start:start + BATCH_SIZE])


def validation_loss(pairs=500):
    model.eval()
    held_out, total = val[:pairs], 0.0
    with torch.no_grad(), amp():
        for start in range(0, len(held_out), BATCH_SIZE):
            chunk = held_out[start:start + BATCH_SIZE]
            total += model(**batch_of(chunk)).loss.item() * len(chunk)
    model.train()
    return total / len(held_out)


bf16 = device.type == "cuda" and torch.cuda.is_bf16_supported()
amp = (lambda: torch.autocast("cuda", dtype=torch.bfloat16)) if bf16 else contextlib.nullcontext
tokenizer.src_lang, tokenizer.tgt_lang = "eng_Latn", TARGET_CODE
torch.manual_seed(SEED)
# A fixed learning rate with warmup, as A8's; Adafactor's own schedule and scaling are off.
optimizer = Adafactor(model.parameters(), lr=LEARNING_RATE, scale_parameter=False,
                      relative_step=False, warmup_init=False)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: min(1.0, (step + 1) / WARMUP))
checkpointer = TrainingCheckpointer(f"tutorial9-{LANGUAGE}-{TARGET_CODE}",
                                    checkpoint_dir=OUT_DIR / "checkpoints", save_every_seconds=600)
# Two checkpoints of about 2.5 GB, and a third while one is being written; less when resuming.
needed_gb = 3 if checkpointer.has_checkpoint() else 8
free_gb = shutil.disk_usage(OUT_DIR).free / 1e9
if free_gb < needed_gb:
    raise SystemExit(f"Your Drive has {free_gb:.1f} GB free and the checkpoints need about "
                     f"{needed_gb} GB. Free some space (Drive's trash counts until emptied), "
                     "then run this cell again.")
if checkpointer.has_checkpoint():
    checkpointer.load(model, optimizer, scheduler, map_location=device)
state = checkpointer.state

model.train()
started, first_step, losses = time.monotonic(), state.global_step, []
while state.global_step < STEPS:
    step_loss = 0.0
    for a in range(ACCUMULATE):
        with amp():
            loss = model(**micro_batch(state.global_step * ACCUMULATE + a)).loss / ACCUMULATE
        loss.backward()
        step_loss += loss.item()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()
    state.global_step += 1
    losses.append(step_loss)
    done = state.global_step - first_step
    if done == 20:
        per_step = (time.monotonic() - started) / done
        left = (STEPS - state.global_step) * per_step / 60
        print(f"{per_step:.2f} s a step: about {left:.0f} more minutes for {STEPS:,} steps.")
    if state.global_step % EVAL_EVERY == 0 or state.global_step == STEPS:
        val_loss = validation_loss()
        best = val_loss < state.best_val_loss
        state.val_losses.append(val_loss)
        state.best_val_loss = min(state.best_val_loss, val_loss)
        print(f"step {state.global_step:5,}   training loss {sum(losses) / len(losses):.3f}   "
              f"validation loss {val_loss:.3f}{'   best so far' if best else ''}")
        losses = []
        checkpointer.save(model, optimizer, scheduler, is_best=best)
    elif checkpointer.should_save():
        checkpointer.save(model, optimizer, scheduler)

if checkpointer.has_checkpoint("best"):
    checkpointer.load(model, which="best", map_location=device, restore_rng=False)
print(f"Fine-tuned for {state.global_step:,} steps; using the best checkpoint, validation loss "
      f"{state.best_val_loss:.3f}. Losses checked: {[round(v, 3) for v in state.val_losses]}")

## Step 7: what fine-tuning bought

The fine-tuned model translates the same test set, decoded the same two ways, and the report
puts every system side by side: NLLB off the shelf, NLLB fine-tuned, and your A9 model, trained
from scratch on the same pairs. Then a few test sentences where fine-tuning changed the
output, so you can see *what* changed, not only by how much.

In [ ]:
fine_tuned = {how: decode("fine-tuned", how, reuse=False) for how in DECODINGS}
for how, hyps in fine_tuned.items():
    score("fine-tuned", how, hyps)

report = pd.DataFrame(scores)
report.insert(0, "language", LANGUAGE)
report.to_csv(OUT_DIR / "scores.tsv", sep="\t", index=False)
print("\n" + report[["system", "decoding", "bleu", "chrf"]].to_string(index=False))
print(f"\nBLEU signature: {scores[0]['bleu_signature']}")
print(f"Target code: {TARGET_CODE} ({REASON.strip()})")
print(f"Your prediction: {PREDICTION.strip()}")

print("\nWhere fine-tuning changed the greedy output (the first five):")
changed = [i for i, (a, b) in enumerate(zip(off_the_shelf["greedy"], fine_tuned["greedy"])) if a != b]
for i in changed[:5]:
    print(f"\n  English:        {sources[i]}\n  reference:      {references[i]}")
    print(f"  off the shelf:  {off_the_shelf['greedy'][i]}\n  fine-tuned:     {fine_tuned['greedy'][i]}")
print(f"\n{len(changed):,} of {len(sources):,} greedy translations changed.")

record = {"language": LANGUAGE, "target_code": TARGET_CODE, "reason": REASON.strip(),
          "model": NLLB_MODEL, "steps": state.global_step, "batch_size": BATCH_SIZE,
          "accumulate": ACCUMULATE, "learning_rate": LEARNING_RATE, "warmup": WARMUP,
          "seed": SEED, "best_val_loss": state.best_val_loss, "bf16": bf16,
          "prediction": PREDICTION.strip(), "torchlingo": torchlingo.version_label()}
(OUT_DIR / "t9-run.json").write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
print(f"Saved scores.tsv, the .hyp files and t9-run.json in {OUT_DIR}")


## What to take from this

Questions to answer for yourself, from your own table:

1. **What did fine-tuning buy over the model off the shelf?** Put your prediction beside the
   result. For a borrowed code, how much of the off-the-shelf score was the borrowed
   language, and how much of yours does the fine-tuned output now look like?
2. **Fine-tuned NLLB against your A9 model.** Both saw your pairs; one started from 200
   languages and the other from nothing. Which won, by how much, and what does the gap say
   about where a model's knowledge of your language came from?
3. **Read the five changed sentences.** What did fine-tuning change: words, spelling, word
   order, the register of the Church's texts? Is any change for the worse?
4. **The language code.** Why did the model need a target code at all, and what did fine-tuning
   have to unlearn about the one it was given?
5. **What would you change for a Final Project?** More steps, a different learning rate, both
   directions with two codes, as A14 does with tags? Say what result would tell you it
   worked.

The validation losses the training cell printed show whether more steps would have helped:
still falling at the end means the budget, not the model, set the result.

---
## When you finish: disconnect the GPU

**Runtime → Disconnect and delete runtime.** Colab charges compute units for every minute a GPU runtime stays connected, even when nothing is running, so a runtime left open after the work is done uses up your monthly units. Closing the browser tab does not end it right away. Everything this notebook writes is in `MyDrive/CS479/tutorial9`, so disconnecting loses nothing.